# Analyse du Chiffre d'Affaires — `resultats_final.csv`
> Visualisations interactives avec **Plotly**.

In [ ]:
# !pip install plotly pandas openpyxl

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TEMPLATE = 'plotly_white'
ACCENT   = '#ef4444'
print('Imports OK')

Imports OK


In [ ]:
df = pd.read_csv(r'c:\Matière logiciels\VS code\ex-git-prenom\taille_100.csv')

# Vrais noms de colonnes
# ['ID', 'Prix', 'Quantite', 'Remise', 'CA_Brut', 'CA_Net', 'TVA', 'CA_Total']

df['ID'] = df['ID'].astype(str)
df['taux_marge'] = (df['TVA'] / df['CA_Net'] * 100).round(1)
df['remise_eur'] = (df['CA_Brut'] - df['CA_Net']).round(2)

print(f' {len(df)} produits chargés')
df


49 produits chargés


,ID,Prix,Quantite,Remise,CA_Brut,CA_Net,TVA,CA_Total,taux_marge,remise_eur
0,100,9.88,4,0.0,39.52,39.52,7.90,47.42,20.0,0.00
1,101,178.98,5,0.0,894.90,894.90,178.98,1073.88,20.0,0.00
2,102,50.37,9,30.0,453.33,317.33,63.47,380.80,20.0,136.00
3,103,86.81,5,0.0,434.05,434.05,86.81,520.86,20.0,0.00
4,104,36.13,5,0.0,180.65,180.65,36.13,216.78,20.0,0.00
5,105,79.09,6,0.0,474.54,474.54,94.91,569.45,20.0,0.00
6,106,109.56,7,0.0,766.92,766.92,153.38,920.30,20.0,0.00
7,107,127.58,6,0.0,765.48,765.48,153.10,918.58,20.0,0.00
8,108,119.95,5,5.0,599.75,569.76,113.95,683.71,20.0,29.99
9,109,79.12,14,10.0,1107.68,996.91,199.38,1196.29,20.0,110.77


---
## 1 · CA Total par produit

In [ ]:
df_sorted = df.sort_values('CA_Total', ascending=False)

fig = px.bar(
    df_sorted,
    x='ID', y='CA_Total',
    color='CA_Total',
    color_continuous_scale='Blues',
    text=df_sorted['CA_Total'].map(lambda v: f'{v:,.0f} €'),
    labels={'ID': 'ID Produit', 'CA_Total': 'CA Total (€)'},
    title='Chiffre d\'affaires total par produit',
    template=TEMPLATE
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.update_layout(
    coloraxis_showscale=False,
    title_font_size=18,
    height=480,
    yaxis=dict(gridcolor='#f0f0f0')
)
fig.show()

---
## 2 · CA Brut vs CA Net — impact des remises

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name='CA Net',
    x=df['ID'], y=df['CA_Net'],
    marker_color='#2563eb',
    hovertemplate='<b>Produit %{x}</b><br>CA net : %{y:,.0f} €<extra></extra>'
))
fig.add_trace(go.Bar(
    name='Remise appliquée',
    x=df['ID'], y=df['remise_eur'],
    marker_color=ACCENT, opacity=0.75,
    hovertemplate='<b>Produit %{x}</b><br>Remise : %{y:,.0f} €<extra></extra>'
))

fig.update_layout(
    barmode='stack',
    title='CA Brut décomposé : CA Net + Remises',
    title_font_size=18,
    xaxis_title='ID Produit', yaxis_title='CA (€)',
    template=TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    yaxis=dict(gridcolor='#f0f0f0')
)
fig.show()

---
## 3 · TVA par produit

In [ ]:
df_t = df.sort_values('TVA', ascending=True)

fig = go.Figure(go.Bar(
    x=df_t['TVA'], y=df_t['ID'],
    orientation='h',
    marker_color='#2563eb',
    text=df_t['TVA'].map(lambda v: f'{v:,.0f} €'),
    textposition='outside',
    hovertemplate='<b>Produit %{y}</b><br>TVA : %{x:,.0f} €<extra></extra>'
))
fig.update_layout(
    title='TVA collectée par produit',
    title_font_size=18,
    xaxis_title='TVA (€)', yaxis_title='ID Produit',
    template=TEMPLATE, height=520,
    xaxis=dict(gridcolor='#f0f0f0')
)
fig.show()

---
## 4 · Scatter : CA Net vs TVA (bulle = quantité, couleur = remise)

In [ ]:
fig = px.scatter(
    df,
    x='CA_Net', y='TVA',
    size='Quantite', color='Remise',
    color_continuous_scale='RdYlGn_r',
    hover_name='ID',
    hover_data={'CA_Net': ':,.0f', 'TVA': ':,.0f', 'Quantite': True, 'Remise': True},
    labels={'CA_Net': 'CA Net (€)', 'TVA': 'TVA (€)', 'Remise': 'Remise (%)', 'Quantite': 'Quantité'},
    title='Relation TVA / CA Net — taille = quantité, couleur = remise',
    template=TEMPLATE
)
fig.update_traces(marker_line_width=0.5, marker_line_color='white')
fig.update_layout(title_font_size=18, height=500)
fig.show()

---
## 5 · Dashboard récapitulatif

In [ ]:
ca_total   = df['CA_Total'].sum()
tva_tot    = df['TVA'].sum()
remise_tot = df['remise_eur'].sum()
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Répartition CA Total (top 10)',
        'Prix unitaire vs Quantité vendue',
        'Distribution des remises (%)',
        'CA Net vs TVA par produit'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'xy'}]
    ],
    vertical_spacing=0.18, horizontal_spacing=0.12
)
# Pie CA total
top10 = df.nlargest(10, 'CA_Total')
fig.add_trace(go.Pie(
    labels=top10['ID'], values=top10['CA_Total'],
    hole=0.4, textinfo='label+percent', showlegend=False
), row=1, col=1)

# Scatter prix / quantité
fig.add_trace(go.Scatter(
    x=df['Prix'], y=df['Quantite'],
    mode='markers+text', text=df['ID'],
    textposition='top center', textfont_size=8,
    marker=dict(color='#2563eb', size=9, opacity=0.75),
    hovertemplate='Produit %{text}<br>Prix : %{x:.2f} €<br>Qté : %{y}<extra></extra>'
), row=1, col=2)

# Histogram remises
fig.add_trace(go.Histogram(
    x=df['Remise'], nbinsx=8,
    marker_color='#3b82f6', opacity=0.8
), row=2, col=1)

# Bar CA Net vs TVA
fig.add_trace(go.Bar(
    name='CA Net', x=df['ID'], y=df['CA_Net'],
    marker_color='#2563eb', showlegend=False
), row=2, col=2)
fig.add_trace(go.Bar(
    name='TVA', x=df['ID'], y=df['TVA'],
    marker_color='#10b981', showlegend=False
), row=2, col=2)

fig.update_layout(
    title=dict(
        text=f'Dashboard CA — Total : {ca_total:,.0f} € | TVA : {tva_tot:,.0f} € | Remises : {remise_tot:,.0f} €',
        font_size=14
    ),
    template=TEMPLATE,
    height=780,
    barmode='group'
)
fig.show()

---
## 6 · Export (optionnel)

In [ ]:
fig.write_html(r'c:\Matière logiciels\VS code\ex-git-prenom\dashboard_ca.html')
print('Export terminé !')

Export terminé !
